## Phase 2: Pairwise Similarity Computation & Creating Nodes/Edges
#### File: similarity_computation.ipynb

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import euclidean_distances

In [2]:
sample = pd.read_csv('../data/spotify_sample.csv')
sample = sample.reset_index(drop=True)

n = len(sample)
print(f'Total songs loaded: {n}')
print(f'Total pairs to evaluate: {n*(n-1)//2:,}')


Total songs loaded: 2506
Total pairs to evaluate: 3,138,765


#### Feature engineering and weighting is based on sonic importance according to a Frontiers in Psychology paper: “Acoustic Features Influence Musical Choices Across Multiple Genres” (https://www.frontiersin.org/journals/psychology/articles/10.3389/fpsyg.2017.00931/full)


In [3]:
# feature engineering to create more sonic separation between songs, enabling more meaningful similarity scoring

# 1. Acoustic-Electric spectrum
#    high acousticness + low energy = folk/acoustic
#    low acousticness + high energy = rock/electronic
sample['acoustic_electric'] = sample['acousticness'] - sample['energy']

# 2. Mood index: combines valence and energy into emotional quadrants
#    High energy + high valence = euphoric (party music)
#    High energy + low valence = angry/intense (punk, metal)
#    Low energy + high valence = peaceful/happy (acoustic pop)
#    Low energy + low valence = melancholic (sad indie, classical)
sample['mood_index'] = sample['valence'] * sample['energy']

# 3. Vocal prominence: speechiness + (1 - instrumentalness)
#    High = very vocal/rap, Low = instrumental
sample['vocal_prominence'] = sample['speechiness'] + (1 - sample['instrumentalness'])

# 4. Rhythmic intensity: danceability * tempo (normalized)
#    Captures whether a song has strong driving rhythm
sample['rhythmic_intensity'] = sample['danceability'] * (sample['tempo'] / sample['tempo'].max())

# 5. Sonic density: loudness normalized + energy
#    Captures how "full" and "heavy" a song sounds
sample['sonic_density'] = (
    (sample['loudness'] - sample['loudness'].min()) /
    (sample['loudness'].max() - sample['loudness'].min())
) + sample['energy']


In [4]:
FEATURE_WEIGHTS = {
    # Derived features
    'mood_index':         3.0,  # emotional character is most important
    'acoustic_electric':  2.5,  # electric vs organic texture
    'vocal_prominence':   2.0,  # vocal vs instrumental character
    'rhythmic_intensity': 2.0,  # rhythm feel
    'sonic_density':      1.5,  # heaviness/fullness of sound
    # Raw features
    'valence':            1.5,  # emotional tone
    'energy':             1.5,  # raw energy level
    'danceability':       1.0,  # groove
    'acousticness':       1.0,  # texture
    'tempo':              0.5,  # BPM
}

In [5]:
features = sample[list(FEATURE_WEIGHTS.keys())].values.astype(float)
weights = np.array(list(FEATURE_WEIGHTS.values()))
features_weighted = features * weights

print(f'\nFeature matrix shape: {features_weighted.shape}')
print(f'Sample of weighted features (first 3 rows):\n{features_weighted[:3]}')


Feature matrix shape: (2506, 10)
Sample of weighted features (first 3 rows):
[[ 1.48350887 -1.84409034  1.8867526   0.93939018  2.37266343  0.99033571
   1.12348938  0.61260331  0.01135678  0.3833599 ]
 [ 0.26402233  1.86740276  0.51437006  0.57171342  1.23130048  0.54781282
   0.36146789  0.56198347  0.9879397   0.2543284 ]
 [ 0.18401138 -1.21757787  2.06573077  0.630263    2.32483862  0.13550356
   1.01848642  0.55578512  0.1919598   0.2835012 ]]


In [6]:
# normalize after weighting so that the scale differences introduced by weights don't distort Euclidean distances

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features_weighted)

In [7]:
# compute full Euclidean distance matrix (choose Euclidean distance over cosine similarity only measures the angle between vectors, ignoring magnitude differences)
# return: (n x n) matrix where entry [i,j] is the Euclidean distance between song i and song j

dist_matrix = euclidean_distances(features_scaled)

In [8]:
# convert distance to similarity score in [0, 1]:
sim_matrix = 1 / (1 + dist_matrix)

print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'Sample similarity scores (first 3x3):\n{sim_matrix[:3,:3].round(4)}')

Similarity matrix shape: (2506, 2506)
Sample similarity scores (first 3x3):
[[1.     0.3901 0.548 ]
 [0.3901 1.     0.4432]
 [0.548  0.4432 1.    ]]


In [9]:
sample_indices = np.random.choice(n, size=min(1000, n), replace=False)
sample_scores = sim_matrix[np.ix_(sample_indices, sample_indices)].flatten()
sample_scores = sample_scores[sample_scores < 0.9999]  # exclude diagonal (self-similarity)

print(f'\nScore distribution:')
print(f'  Mean:              {sample_scores.mean():.4f}')
print(f'  Median:            {np.median(sample_scores):.4f}')
print(f'  75th percentile:   {np.percentile(sample_scores, 75):.4f}')
print(f'  90th percentile:   {np.percentile(sample_scores, 90):.4f}')
print(f'  95th percentile:   {np.percentile(sample_scores, 95):.4f}')
print(f'  99th percentile:   {np.percentile(sample_scores, 99):.4f}')

print()
for thresh in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    pct = (sample_scores >= thresh).mean()
    estimated_edges = int(pct * (n * (n-1) / 2))
    print(f'  Threshold {thresh}: ~{estimated_edges:,} edges')


Score distribution:
  Mean:              0.5460
  Median:            0.5405
  75th percentile:   0.6172
  90th percentile:   0.6876
  95th percentile:   0.7298
  99th percentile:   0.8037

  Threshold 0.65: ~530,005 edges
  Threshold 0.7: ~260,349 edges
  Threshold 0.75: ~106,575 edges
  Threshold 0.8: ~34,429 edges
  Threshold 0.85: ~7,609 edges
  Threshold 0.9: ~816 edges
  Threshold 0.95: ~12 edges


In [10]:
# Threshold check: if a song has 0 neighbors, use the fallback threshold

PRIMARY_THRESHOLD = 0.70
FALLBACK_THRESHOLD = 0.65

strokes_indices = sample[sample['artist'].str.contains('The Strokes', na=False)].index.tolist()
regina_indices = sample[sample['artist'].str.contains('Regina Spektor', na=False)].index.tolist()
anchor_indices = strokes_indices + regina_indices

print(f'\nStrokes songs in sample: {len(strokes_indices)}')
print(f'Regina songs in sample:  {len(regina_indices)}')
print(f'\nConnectivity check at threshold {PRIMARY_THRESHOLD}:')
print(f'{'Song':<40} {'Artist':<20} {'Neighbors'}')
print('-' * 75)

isolated_indices = set()
for idx in range(n):
    neighbor_count = (sim_matrix[idx] >= PRIMARY_THRESHOLD).sum() - 1
    if neighbor_count == 0:
        isolated_indices.add(idx)

for idx in anchor_indices:
    row = sample.iloc[idx]
    neighbor_count = (sim_matrix[idx] >= PRIMARY_THRESHOLD).sum() - 1
    flag = ' ← ISOLATED' if neighbor_count == 0 else ''
    print(f"{row['track_name']:<40} {row['artist']:<20} {neighbor_count}{flag}")

print(f'\nTotal isolated songs across full sample: {len(isolated_indices)}')
print(f'These will use fallback threshold of {FALLBACK_THRESHOLD}')


Strokes songs in sample: 43
Regina songs in sample:  3

Connectivity check at threshold 0.7:
Song                                     Artist               Neighbors
---------------------------------------------------------------------------
The Adults Are Talking                   The Strokes          391
Call It Fate, Call It Karma              The Strokes          44
Selfless                                 The Strokes          202
Last Nite                                The Strokes          343
Reptilia                                 The Strokes          33
Ode To The Mets                          The Strokes          121
Why Are Sundays So Depressing            The Strokes          428
Someday                                  The Strokes          218
You Only Live Once                       The Strokes          79
Eternal Summer                           The Strokes          408
Is This It                               The Strokes          372
Bad Decisions                      

In [11]:
# extract edges using hybrid threshold 
edges = []
ids = sample['track_id'].values

for i in range(n):
    threshold_i = FALLBACK_THRESHOLD if i in isolated_indices else PRIMARY_THRESHOLD

    for j in range(i + 1, n):
        threshold_j = FALLBACK_THRESHOLD if j in isolated_indices else PRIMARY_THRESHOLD
        effective_threshold = min(threshold_i, threshold_j)

        score = sim_matrix[i, j]
        if score >= effective_threshold:
            edges.append({'source_id': ids[i], 'target_id': ids[j],'score': round(float(score), 4)})

In [12]:
edges_df = pd.DataFrame(edges)
print(f'Total edges extracted: {len(edges_df):,}')

Total edges extracted: 243,777


In [13]:
sample.to_csv('nodes.csv', index=False)
edges_df.to_csv('edges.csv', index=False)

print(f'nodes.csv: {len(sample):,} rows')
print(f'edges.csv: {len(edges_df):,} rows')

nodes.csv: 2,506 rows
edges.csv: 243,777 rows


In [14]:
strokes_ids = set(sample[sample['artist'].str.contains('The Strokes', na=False)]['track_id'])
regina_ids  = set(sample[sample['artist'].str.contains('Regina Spektor', na=False)]['track_id'])

strokes_edges = edges_df[
    edges_df['source_id'].isin(strokes_ids) |
    edges_df['target_id'].isin(strokes_ids)
]
regina_edges = edges_df[
    edges_df['source_id'].isin(regina_ids) |
    edges_df['target_id'].isin(regina_ids)
]

print(f'\nEdges connected to Strokes songs: {len(strokes_edges):,}')
print(f'Edge connected to Regina songs:  {len(regina_edges):,}')
print(f'\nSample of Strokes edges:')
strokes_edges.head(10)


Edges connected to Strokes songs: 8,996
Edge connected to Regina songs:  378

Sample of Strokes edges:


,source_id,target_id,score
0,5ruzrDWcT0vuJIOMW7gMnW,57Xjny5yNzAcsxnusKmAfA,0.7151
1,5ruzrDWcT0vuJIOMW7gMnW,6u0x5ad9ewHvs3z6u9Oe3c,0.7265
2,5ruzrDWcT0vuJIOMW7gMnW,7bfocP7GYoLOutUYpTI8tx,0.7883
3,5ruzrDWcT0vuJIOMW7gMnW,4ileLT7ldd2uX8bMemWqbm,0.7517
4,5ruzrDWcT0vuJIOMW7gMnW,7G8hUONVhvJnkD3Ak8mNF1,0.7666
5,5ruzrDWcT0vuJIOMW7gMnW,2ThdB23G9Rgf1ExndFGEEg,0.7572
6,5ruzrDWcT0vuJIOMW7gMnW,37ZgsOy8t4vMnGtMExr6ah,0.7631
7,5ruzrDWcT0vuJIOMW7gMnW,37cb7mzkQKksGm0noGDXeh,0.7962
8,5ruzrDWcT0vuJIOMW7gMnW,0J2OnBNKwt0KICDyDFPUvl,0.7070
9,5ruzrDWcT0vuJIOMW7gMnW,72R0X0h8YaxYNpegeoOl0M,0.7497
